# 🦜 LangChain Prompt Templates — Comprehensive Revision Guide

This notebook is a **structured revision reference** for LangChain's prompt-templating system.
It is organised by concept (not by file) so the ideas build on each other coherently.

---
### 📚 Table of Contents

1. [What is a Prompt Template?](#1-what-is-a-prompt-template)
2. [PromptTemplate — String Templates](#2-prompttemplate)
3. [Template Properties & Introspection](#3-template-properties)
4. [Template Methods — invoke, format, batch, save/load](#4-template-methods)
5. [ChatPromptTemplate — Multi-turn / Role-based Prompts](#5-chatprompttemplate)
6. [Message Types — HumanMessage, SystemMessage, AIMessage](#6-message-types)
7. [Few-Shot Prompting with `FewShotPromptTemplate`](#7-fewshot-string)
8. [Few-Shot Chat Prompting with `FewShotChatMessagePromptTemplate`](#8-fewshot-chat)
9. [Save & Load Prompt Templates (JSON)](#9-save-load)
10. [End-to-End Pattern Reference & Mental Model](#10-pattern-reference)

---


## 0. Setup

All imports and the shared model used throughout the notebook.

In [ ]:
# pip install langchain langchain-groq python-dotenv  (run once)

import os
from dotenv import load_dotenv
load_dotenv()   # reads GROQ_API_KEY from .env

from langchain_groq import ChatGroq
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    FewShotPromptTemplate,
    FewShotChatMessagePromptTemplate,
)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Shared model — used wherever LLM responses are shown
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.5)
print("Setup complete ✓")


---
## 1. What is a Prompt Template?

### 🧠 Intuition
A **Prompt Template** is a reusable blueprint for constructing prompts.
Instead of hard-coding a string each time, you define a template with
**placeholder variables** (`{like_this}`) and fill them in at runtime.

```
Template:  "Who is the {role} of {country}?"
Inputs:    role="President", country="India"
Output:    "Who is the President of India?"
```

### Why use them instead of f-strings?

| Raw f-string | Prompt Template |
|---|---|
| No validation | Validates required variables |
| Not serialisable | Save / load as JSON |
| Not composable | Works inside LCEL chains with `\|` |
| No `.batch()` | Built-in batch invocation |
| No `.pretty_print()` | Human-readable debug view |

### The Template Hierarchy

```
BasePromptTemplate
├── PromptTemplate          ← plain string template
└── BaseChatPromptTemplate
    └── ChatPromptTemplate  ← list of (role, content) messages
```


---
## 2. `PromptTemplate` — String Templates

### 🧠 Intuition
`PromptTemplate` handles **single-string** prompts.
Use it when you are calling a plain-text completion model,
or when you need a compact, single-turn instruction.

### Two ways to create one


In [ ]:
from langchain_core.prompts import PromptTemplate

# ── 2a. from_template() — most common, minimal boilerplate ──────────────────
prompt_a = PromptTemplate.from_template("Who is the {role} of {country}?")
print("2a input_variables:", prompt_a.input_variables)   # ['role', 'country']

# ── 2b. Constructor — explicit, useful when you need extra control ───────────
prompt_b = PromptTemplate(
    template="Who is the {role} of {country}?",
    input_variables=["role", "country"],
    input_types={"role": str, "country": str},  # optional type hints
    validate_template=True,                      # raises error if vars mismatch
)
print("2b template_format:", prompt_b.template_format)   # 'f-string' (default)

# ── 2c. Invoke to get a PromptValue ─────────────────────────────────────────
result = prompt_a.invoke({"role": "Prime Minister", "country": "India"})
print("2c invoked:", result)
print("    type  :", type(result).__name__)   # StringPromptValue


### 📝 Key Points
- `from_template()` auto-detects `input_variables` from `{...}` placeholders.
- Default `template_format` is `"f-string"`. You can also use `"jinja2"` for more
  complex templating (loops, conditionals).
- `.invoke()` returns a **`PromptValue`** object (not a plain string) so it can flow
  directly into the next step of an LCEL chain.


---
## 3. Template Properties & Introspection

### 🧠 Intuition
Every `PromptTemplate` (and `ChatPromptTemplate`) exposes metadata about itself.
Knowing these properties lets you audit, validate, and debug templates programmatically.


In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# ── 3a. PromptTemplate properties ───────────────────────────────────────────
pt = PromptTemplate(
    template="Who is the {role} of {country}?",
    input_variables=["role", "country"],
    validate_template=True,
)

print("── PromptTemplate ──────────────────────────────────")
print("template         :", pt.template)
print("template_format  :", pt.template_format)   # 'f-string'
print("input_variables  :", pt.input_variables)   # ['role', 'country']
print("input_types      :", pt.input_types)
print("optional_variables:", pt.optional_variables)
print("validate_template:", pt.validate_template)

# ── 3b. ChatPromptTemplate properties ───────────────────────────────────────
cpt = ChatPromptTemplate(
    messages=[
        ("system", "You are a helpful assistant."),
        ("user",   "How do I bake {pastry}?"),
    ]
)

print("\n── ChatPromptTemplate ──────────────────────────────")
print("input_variables  :", cpt.input_variables)   # ['pastry']
print("messages count   :", len(cpt.messages))

# ── 3c. pretty_print — human-readable visual of the template ─────────────────
print("\n── pretty_print() output ───────────────────────────")
pt.pretty_print()


### 📝 Key Points
- `input_variables` — the list of **required** `{placeholders}` in the template.
- `optional_variables` — placeholders with default values (set via `partial_variables`).
- `validate_template=True` — raises `ValueError` at construction time if declared
  `input_variables` don't match the actual placeholders in the template string.
- `pretty_print()` is invaluable during development — it shows exactly what the
  model will see, with coloured placeholder highlighting in supported terminals.


---
## 4. Template Methods — `invoke`, `format`, `format_prompt`, `batch`, `save`

### 🧠 Intuition
A `PromptTemplate` gives you **multiple ways** to materialise a prompt from variables.
They all produce the same final text, but return different types — important when
you need to plug the result into different downstream tools.

| Method | Input | Output type | Use when… |
|---|---|---|---|
| `.invoke(dict)` | dict | `PromptValue` | feeding into LCEL chain |
| `.format(**kwargs)` | kwargs | `str` | you just want the plain string |
| `.format_prompt(**kwargs)` | kwargs | `PromptValue` | same as invoke, older API |
| `.batch([dict, ...])` | list of dicts | list of `PromptValue` | many inputs at once |
| `.save(path)` | — | JSON file | persisting/sharing templates |


In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("Who is the {role} of {country}?")

# ── 4a. invoke — returns PromptValue (LCEL-compatible) ──────────────────────
pv = prompt.invoke({"role": "Prime Minister", "country": "India"})
print("4a invoke (PromptValue)  :", pv)
print("   to_string()           :", pv.to_string())

# ── 4b. format — returns plain str ──────────────────────────────────────────
s = prompt.format(role="Prime Minister", country="India")
print("\n4b format (str)         :", s)
print("   type                  :", type(s).__name__)

# ── 4c. format_prompt — returns PromptValue (older alias for invoke) ─────────
fp = prompt.format_prompt(role="President", country="USA")
print("\n4c format_prompt        :", fp)

# ── 4d. batch — list of dicts → list of PromptValues ────────────────────────
batch_results = prompt.batch([
    {"role": "Prime Minister", "country": "India"},
    {"role": "President",      "country": "USA"},
    {"role": "President",      "country": "China"},
])
print("\n4d batch results:")
for r in batch_results:
    print("  ", r.to_string())

# ── 4e. pretty_print — visual debug view ────────────────────────────────────
print("\n4e pretty_print:")
prompt.pretty_print()

# ── 4f. save — serialise to JSON ─────────────────────────────────────────────
prompt.save("/tmp/my_prompt.json")
print("\n4f saved to /tmp/my_prompt.json")
import json
with open("/tmp/my_prompt.json") as f:
    print(json.dumps(json.load(f), indent=2))


### 📝 Key Points
- `.format()` → `str` is the escape hatch when you need a raw string (e.g. logging).
- `.invoke()` → `PromptValue` is preferred inside chains because it carries type info
  that lets LangChain route it correctly to a chat model vs. a text model.
- `.batch()` does **not** parallelise LLM calls — it only fills in template variables
  for many inputs at once. Parallelism happens when the **model** is batched.
- `.save()` serialises everything needed to reconstruct the template — including
  `input_variables`, `template_format`, and optional metadata.


---
## 5. `ChatPromptTemplate` — Multi-turn / Role-based Prompts

### 🧠 Intuition
`ChatPromptTemplate` is the **chat-model equivalent** of `PromptTemplate`.
Instead of a single string, it holds an **ordered list of messages**, each tagged
with a role: `system`, `human`, or `ai`.

```
system  → sets persona / instructions
human   → represents the user's turn
ai      → represents the model's prior turn (for few-shot / multi-turn)
```

This maps directly to the message format expected by OpenAI, Groq, Anthropic, etc.

### Three ways to define messages


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# ── 5a. Tuple shorthand — most concise ──────────────────────────────────────
cpt_a = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Always reply in {language}."),
    ("human",  "Tell me about {topic}."),
])
print("5a input_variables:", cpt_a.input_variables)

invoked_a = cpt_a.invoke({"language": "French", "topic": "black holes"})
print("5a messages:")
for m in invoked_a.messages:
    print(f"  [{m.type:6s}] {m.content[:60]}")

# ── 5b. from_template (single human message) ────────────────────────────────
cpt_b = ChatPromptTemplate.from_template("Translate this to {language}: {text}")
print("\n5b input_variables:", cpt_b.input_variables)

# ── 5c. Constructor with messages list ──────────────────────────────────────
cpt_c = ChatPromptTemplate(
    messages=[
        ("system", "You are a helpful assistant."),
        ("user",   "How do I bake {pastry}?"),
    ]
)
print("5c input_variables:", cpt_c.input_variables)

# ── 5d. Full chain: template → model → response ─────────────────────────────
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

model = ChatGroq(model="llama-3.3-70b-versatile")
chain = cpt_a | model | StrOutputParser()
result = chain.invoke({"language": "English", "topic": "photosynthesis"})
print("\n5d chain result:\n", result[:200], "...")


### 📝 Key Points
- Tuple format `("role", "content_with_{vars}")` is the cleanest syntax.
- Valid role strings: `"system"`, `"human"` / `"user"`, `"ai"` / `"assistant"`.
- `.invoke()` returns a `ChatPromptValue` whose `.messages` is a list of
  `SystemMessage`, `HumanMessage`, `AIMessage` objects.
- `ChatPromptTemplate` is also a `Runnable` — pipe it directly into a model with `|`.


---
## 6. Message Types — `HumanMessage`, `SystemMessage`, `AIMessage`

### 🧠 Intuition
Under the hood, LangChain represents every turn in a conversation as a
typed **Message object**. You rarely construct these manually — `ChatPromptTemplate`
does it for you — but understanding them helps when:
- Inspecting what a template produces
- Building dynamic multi-turn conversation history
- Debugging role assignment issues

| Class | Role string | When to use |
|---|---|---|
| `SystemMessage` | `"system"` | Global instructions / persona |
| `HumanMessage` | `"human"` / `"user"` | User's input |
| `AIMessage` | `"ai"` / `"assistant"` | Model's prior reply (few-shot, history) |


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

# ── 6a. Constructing messages manually ──────────────────────────────────────
sys_msg   = SystemMessage(content="You are a pirate. Always speak like one.")
user_msg  = HumanMessage(content="What is the capital of France?")
ai_msg    = AIMessage(content="Arr, 'tis Paris, the jewel of the seven seas!")

print("6a types:")
for m in [sys_msg, user_msg, ai_msg]:
    print(f"  {type(m).__name__:15s} | role={m.type:10s} | {m.content[:40]}")

# ── 6b. What ChatPromptTemplate produces under the hood ─────────────────────
template = ChatPromptTemplate.from_messages([
    ("system", "Always reply in {number_of_lines} lines."),
    ("human",  "Write a paragraph about {topic}."),
])
prompt_value = template.invoke({"number_of_lines": "3", "topic": "gravity"})

print("\n6b ChatPromptTemplate output messages:")
for m in prompt_value.messages:
    print(f"  {type(m).__name__:15s} | {m.content}")

# ── 6c. Manually build a multi-turn conversation and pass to model ────────────
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.3-70b-versatile")

conversation = [
    SystemMessage(content="You are a helpful cooking assistant."),
    HumanMessage(content="How do I make pasta?"),
    AIMessage(content="Boil water, cook pasta 8-10 min, drain, add sauce."),
    HumanMessage(content="What sauce do you recommend?"),   # new turn
]
response = model.invoke(conversation)
print("\n6c Multi-turn response:", response.content[:150], "...")


### 📝 Key Points
- `m.type` gives the role string: `"system"`, `"human"`, `"ai"`.
- `m.content` is always a string (or list of content blocks for multimodal).
- For dynamic conversation history (chatbots), build a Python `list` of message
  objects and pass the whole list to `model.invoke()`.
- `MessagesPlaceholder` (not covered here but worth knowing) lets you inject
  a dynamic list of messages into a `ChatPromptTemplate` at the right position.


---
## 7. Few-Shot Prompting — `FewShotPromptTemplate`

### 🧠 Intuition
**Few-shot prompting** is showing the model a handful of input→output examples
before asking it to handle a new input. This guides the model's style, format,
and reasoning without any fine-tuning.

`FewShotPromptTemplate` automates the assembly:

```
[example_1 formatted by example_prompt]
[example_2 formatted by example_prompt]
[example_3 formatted by example_prompt]
[suffix with the actual user query]
```

Use `FewShotPromptTemplate` (this section) for **plain-text / completion models**.
Use `FewShotChatMessagePromptTemplate` (next section) for **chat models**.


In [ ]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ── 7a. example_formatter: how to render each (input, output) pair ───────────
example_formatter = PromptTemplate.from_template(
    "Question: {question}\n{answer}"
)

# ── 7b. The examples — a list of dicts matching the formatter's variables ────
examples = [
    {
        "question": "Tell me about the US",
        "answer":   "Continent: North America | President: Donald Trump | Language: English",
    },
    {
        "question": "Tell me about Spain",
        "answer":   "Continent: Europe | Prime Minister: Pedro Sanchez | Language: Spanish",
    },
    {
        "question": "Tell me about Portugal",
        "answer":   "Continent: Europe | President: Marcelo Rebelo de Sousa | Language: Portuguese",
    },
]

# ── 7c. Assemble FewShotPromptTemplate ──────────────────────────────────────
few_shot = FewShotPromptTemplate(
    example_prompt=example_formatter,   # formats each example dict
    examples=examples,                  # the list of dicts
    suffix="Question: {user_query}",    # appended AFTER all examples
    # prefix="..."                      # optional: prepended BEFORE examples
    # example_separator="\n\n"        # separator between examples (default \n\n)
)

# ── 7d. Inspect what the assembled prompt looks like ────────────────────────
print("7d pretty_print:")
few_shot.pretty_print()
print("\n" + "="*60)

invoked = few_shot.invoke({"user_query": "Tell me about Egypt"})
print("\n7d invoked (raw text):\n", invoked.text[:400])

# ── 7e. Get the LLM response ─────────────────────────────────────────────────
response = model.invoke(invoked)
print("\n7e Model response:\n", response.content)


### 📝 Key Points
- `example_prompt` is just a regular `PromptTemplate` that knows how to render
  **one** example. `FewShotPromptTemplate` calls it once per example and joins the results.
- `suffix` is the actual user query — appended **after** all examples.
- `prefix` (optional) goes **before** all examples — good for a system instruction.
- `example_separator` (default `"\n\n"`) controls spacing between examples.
- The model learns the **format** from the examples implicitly — you don't need to
  explain the format in the suffix.


---
## 8. Few-Shot Chat Prompting — `FewShotChatMessagePromptTemplate`

### 🧠 Intuition
The chat-model version of few-shot prompting. Instead of a flat text block,
examples are rendered as **human/ai message pairs**, which aligns with how
chat models were trained to learn from conversation history.

```
system:  "You are a whimsical mathematician"
human:   "2 ribbit 2"        ← example 1 input
ai:      "4"                 ← example 1 output
human:   "5 ribbit 2"        ← example 2 input
ai:      "10"                ← example 2 output
human:   "3 ribbit 8"        ← actual user question
```

The model infers the pattern from the human/ai pairs.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.5)

# ── 8a. example_formatter: one human+ai pair ────────────────────────────────
example_formatter = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai",    "{output}"),
])

# ── 8b. Examples — list of dicts with 'input' and 'output' keys ─────────────
examples = [
    {"input": "2 ribbit 2", "output": "4"},
    {"input": "5 ribbit 2", "output": "10"},
    {"input": "3 ribbit 3", "output": "4"},   # notice: 3 ribbit 3 = 4 (odd rule!)
]

# ── 8c. Assemble the few-shot block ─────────────────────────────────────────
few_shot_block = FewShotChatMessagePromptTemplate(
    example_prompt=example_formatter,
    examples=examples,
)

# ── 8d. Embed it inside a full ChatPromptTemplate ───────────────────────────
main_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a whimsical mathematician."),
    few_shot_block,             # ← inserts the human/ai example pairs here
    ("human", "{user_prompt}"),
])

# ── 8e. Inspect the assembled message list ──────────────────────────────────
invoked = main_prompt.invoke({"user_prompt": "3 ribbit 8"})
print("8e Assembled messages:")
for m in invoked.messages:
    print(f"  [{m.type:6s}] {m.content}")

# ── 8f. Get the model's response ─────────────────────────────────────────────
chain = main_prompt | model | StrOutputParser()
response = chain.invoke({"user_prompt": "3 ribbit 8"})
print("\n8f Model response:", response)


### 📝 Key Points
- `FewShotChatMessagePromptTemplate` is **not** a full `ChatPromptTemplate` —
  it's a **block** you embed *inside* one, between system and the final human turn.
- The `example_formatter` must produce `human` + `ai` messages. This is what
  teaches the model the pattern as conversation turns.
- `examples` is a plain Python list of dicts — you can generate or filter it
  dynamically (e.g. retrieve semantically similar examples from a vector store
  using `ExampleSelector`).
- The odd rule in this example (`3 ribbit 3 = 4`, `2 ribbit 2 = 4`) shows that
  few-shot can teach **non-standard operations** the model has never seen.


---
## 9. Save & Load Prompt Templates (JSON)

### 🧠 Intuition
LangChain can **serialise** any `PromptTemplate` to a JSON file and **deserialise**
it back. This is useful for:
- Versioning prompts in a repo (prompt as code)
- Sharing templates across projects
- Serving prompts from a config store without code changes

The JSON captures everything: template string, input variables, format, metadata.


In [ ]:
from langchain_core.prompts import PromptTemplate
import json

# ── 9a. Save a template to JSON ──────────────────────────────────────────────
prompt = PromptTemplate.from_template("Who is the {role} of {country}?")
prompt.save("/tmp/saved_prompt.json")

# Inspect the raw JSON
with open("/tmp/saved_prompt.json") as f:
    raw = json.load(f)
print("9a Saved JSON:")
print(json.dumps(raw, indent=2))

# ── 9b. Load it back ────────────────────────────────────────────────────────
from langchain_core.prompts import load_prompt   # utility loader

loaded = load_prompt("/tmp/saved_prompt.json")
print("\n9b Loaded template     :", loaded.template)
print("   input_variables     :", loaded.input_variables)
print("   Same output?        :", loaded.format(role="President", country="USA"))

# ── 9c. Round-trip with ChatPromptTemplate ──────────────────────────────────
# Note: ChatPromptTemplate uses .save() too, but the JSON schema differs
from langchain_core.prompts import ChatPromptTemplate
chat_pt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful."),
    ("human",  "Tell me about {topic}."),
])
chat_pt.save("/tmp/saved_chat_prompt.json")
print("\n9c ChatPromptTemplate JSON:")
with open("/tmp/saved_chat_prompt.json") as f:
    print(json.dumps(json.load(f), indent=2)[:400], "...")


### 📝 Key Points
- `prompt.save("path.json")` works for both `PromptTemplate` and `ChatPromptTemplate`.
- `load_prompt("path.json")` reconstructs the object from JSON.
- The `"_type"` field in the JSON tells LangChain which class to instantiate on load.
- This is **not** a general Python pickle — it's a clean, human-readable format
  that you can edit in a text editor and commit to version control.


---
## 10. End-to-End Pattern Reference & Mental Model

A quick cheat-sheet of every pattern — one cell, ready for fast lookup.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PROMPT TEMPLATE PATTERN REFERENCE
# ════════════════════════════════════════════════════════════════════════════
from langchain_core.prompts import (
    PromptTemplate, ChatPromptTemplate,
    FewShotPromptTemplate, FewShotChatMessagePromptTemplate,
)
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ── Pattern 1: Simple string template ───────────────────────────────────────
pt = PromptTemplate.from_template("Who is the {role} of {country}?")
print("P1:", pt.format(role="PM", country="India"))

# ── Pattern 2: Chat template (system + human) ────────────────────────────────
cpt = ChatPromptTemplate.from_messages([
    ("system", "You are a {persona}."),
    ("human",  "Tell me about {topic}."),
])
msgs = cpt.invoke({"persona": "scientist", "topic": "DNA"}).messages
print("P2 message roles:", [m.type for m in msgs])

# ── Pattern 3: Manual message list → model ───────────────────────────────────
# (for dynamic conversation history)
# model.invoke([SystemMessage("..."), HumanMessage("..."), AIMessage("..."), HumanMessage("...")])

# ── Pattern 4: Few-shot (plain text) ─────────────────────────────────────────
fs = FewShotPromptTemplate(
    example_prompt=PromptTemplate.from_template("Q: {q}\nA: {a}"),
    examples=[{"q": "2+2", "a": "4"}, {"q": "3+3", "a": "6"}],
    suffix="Q: {question}",
)
print("P4:", fs.format(question="5+5"))

# ── Pattern 5: Few-shot chat (human/ai pairs) ────────────────────────────────
fscp = FewShotChatMessagePromptTemplate(
    example_prompt=ChatPromptTemplate.from_messages([("human","{inp}"),("ai","{out}")]),
    examples=[{"inp": "hi", "out": "hello!"}, {"inp": "bye", "out": "goodbye!"}],
)
full = ChatPromptTemplate.from_messages([
    ("system", "You are friendly."),
    fscp,
    ("human", "{user_input}"),
])
print("P5 message count:", len(full.invoke({"user_input": "hey"}).messages))

# ── Pattern 6: Save / Load ───────────────────────────────────────────────────
pt.save("/tmp/ref_prompt.json")
from langchain_core.prompts import load_prompt
loaded = load_prompt("/tmp/ref_prompt.json")
print("P6 loaded:", loaded.format(role="President", country="France"))

# ── Pattern 7: Template in LCEL chain ────────────────────────────────────────
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
chain = cpt | ChatGroq(model="llama-3.3-70b-versatile") | StrOutputParser()
# result = chain.invoke({"persona": "chef", "topic": "pasta"})

print("\nAll patterns compiled ✓  — uncomment chain invocations to test live.")


---
## 🗺️ Mental Model Summary

```
Prompt Template Ecosystem
══════════════════════════

Single string output             List of typed messages
      │                                  │
PromptTemplate                  ChatPromptTemplate
 .from_template("Who is {role}?")  .from_messages([
                                     ("system", "..."),
                                     ("human",  "{q}"),
                                   ])
      │                                  │
 .format(**kw) → str             .invoke(dict) → ChatPromptValue
 .invoke(dict) → PromptValue           │
                                  .messages → [SystemMessage, HumanMessage, ...]

Few-Shot (plain text)           Few-Shot (chat)
      │                                  │
FewShotPromptTemplate           FewShotChatMessagePromptTemplate
  example_prompt = PT             example_prompt = ChatPT([human, ai])
  examples = [dict, ...]          examples = [dict, ...]
  suffix = "Q: {query}"           ↳ embed inside ChatPromptTemplate

Serialisation
  .save("file.json")  ↔  load_prompt("file.json")

LCEL Integration
  PromptTemplate | model | parser      ← all Runnables, composable with |
  ChatPromptTemplate | model | parser
```

---
*Notebook generated for LangChain Prompt Templates revision — Sagar, IISER Berhampur*
